In [ ]:
pip install torch-geometric

In [ ]:
import h5py 
import pandas as pd
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import networkx as nx
from sklearn.preprocessing import StandardScaler
from torch_geometric.utils import dense_to_sparse
from torch_geometric.nn import GATConv
from torch.utils.data import TensorDataset, DataLoader
import math
from math import sqrt as sqrt

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  
with open('/kaggle/input/metr-la-dataset/adj_METR-LA.pkl', 'rb') as f:
    data = pickle.load(f, encoding='latin1')

In [ ]:
node_ids = data[0]  # Nodes ID
node_to_idx = data[1]  # Map ID to an integer index
adj_matrix = data[2]  # Adjacency MAtrix

adj_matrix = torch.FloatTensor(adj_matrix).to(device)
n = 207  
adj_matrix = adj_matrix.view(n, n)

edge_index, _ = dense_to_sparse(adj_matrix)

G = nx.Graph()

#add nodes to the graph
for node_id in node_ids:
    G.add_node(node_id)

#add arcs
for i in range(len(node_ids)):
    for j in range(i+1, len(node_ids)):  # to avoid duplicates, since the arcs are not oriented
        if adj_matrix[i, j] > 0:  
            G.add_edge(node_ids[i], node_ids[j], weight=adj_matrix[i, j])


print("Number of nodes:", G.number_of_nodes())
print("Number of arcs:", G.number_of_edges())

In [ ]:
adj_matrix_symmetric = torch.max(adj_matrix, adj_matrix.T)

# Assert that adjacency matrix is symmetric
assert torch.allclose(adj_matrix_symmetric, adj_matrix_symmetric.T), "Adjacency matrix is not symmetric"

# Convert it in sparse mode
edge_index, edge_attr = dense_to_sparse(adj_matrix_symmetric)

In [ ]:
h5_file_path = "/kaggle/input/metr-la-dataset/METR-LA.h5"

with h5py.File(h5_file_path, 'r') as h5_file:
    if 'df' in h5_file:
        
        if 'axis1' in h5_file['df']: #the timesteps when the values are read from the sensors
            timestamps = h5_file['df']['axis1'][:]
            timestamps = pd.to_datetime(timestamps, unit='ns')
            print("Timestamps:")
            print(timestamps)
        else:
            print("axis1 not found in 'df' groups")

        if 'axis0' in h5_file['df']: #sensors IDs
            sensors = h5_file['df']['axis0'][:]
            sensors = [s.decode('utf-8') for s in sensors] 
            print("Sensors:")
            print(sensors)
        else:
            print("axis0 not found in 'df' group")

       
        if 'block0_values' in h5_file['df']:
            measurements = h5_file['df']['block0_values'][:]
            print("Measurement data shape:", measurements.shape)
            print("Examples of measurements:")
            print(measurements[:5, :5])  # It just prints the elements in the first 5 columns and 5 rows
        else:
            print("block0_values not found in  'df' group")
    else:
        print("Gruop 'df' does not exist in H5")

df = pd.DataFrame(measurements.transpose(), index=sensors, columns=timestamps)
print("DataFrame created.")

In [ ]:
data = df.values
num_sensors = df.shape[0]  
num_timestamps = df.shape[1]

train_percentage = 0.7
val_percentage = 0.15
test_percentage = 0.15

train_size = int(num_timestamps * train_percentage)
val_size = int(num_timestamps * val_percentage)
test_size = num_timestamps - train_size - val_size



train_data = data[:, :train_size]  
val_data = data[:, train_size:train_size + val_size]  
test_data = data[:, train_size + val_size:] 

In [ ]:
def generate_sliding_window_batches(data, input_len, target_len, batch_size):
   
    num_sensors, num_timesteps = data.shape
    total_sequences = num_timesteps - (input_len + target_len) + 1

    # list containing the input window and target window
    inputs = []
    targets = []

    for start in range(total_sequences):
        
        input_window = data[:, start:start + input_len]
        
        target_window = data[:, start + input_len:start + input_len + target_len]

        inputs.append(input_window)
        targets.append(target_window)

    
    inputs = np.array(inputs)  # (num_sequences, 207, input_len)
    targets = np.array(targets)  # (num_sequences, 207, target_len)

    
    num_batches = len(inputs) // batch_size
    batched_inputs = np.array_split(inputs[:num_batches * batch_size], num_batches)
    batched_targets = np.array_split(targets[:num_batches * batch_size], num_batches)

    return batched_inputs, batched_targets

**Model: Bayesian GAT**

In [ ]:
class BayesianGATLayer(torch.nn.Module):
    def __init__(self, in_features, out_features, dropout, heads = 1):
        super().__init__()
        self.gat = GATConv(in_channels = int(in_features), out_channels = int(out_features), heads=heads, dropout=dropout)
        self.log_std = torch.nn.Parameter(torch.Tensor([0.1]))  # Log-variance for uncertainty

    def forward(self, x, edge_index):
        mean = self.gat(x, edge_index)
        std = torch.exp(self.log_std) 
        return mean + std * torch.randn_like(mean)  #bayesian sampling

class BayesianGAT(torch.nn.Module):
    def __init__(self, in_features, hidden_features, out_features, num_layers, heads, dropout):
        super().__init__()
        self.layers = nn.ModuleList()
        
        if(num_layers==1):
            self.layers.append(BayesianGATLayer(in_features, int(out_features), dropout, heads))
        else:
            self.layers.append(BayesianGATLayer(in_features, int(hidden_features), dropout, heads))
            for _ in range(num_layers - 2):
                self.layers.append(BayesianGATLayer(hidden_features * heads, hidden_features, heads=heads, dropout= dropout))
            self.layers.append(BayesianGATLayer(hidden_features * heads, out_features, heads=1, dropout= dropout))

    def forward(self, x, edge_index):
        for layer in self.layers[:-1]: 
            x = torch.relu(layer(x, edge_index))
        x = self.layers[-1](x, edge_index)  
        return x

**Model: Informer**

In [ ]:
class ConvLayer(nn.Module):
    def __init__(self, c_in):
        super(ConvLayer, self).__init__()
        padding = 1 if torch.__version__>='1.5.0' else 2
        self.downConv = nn.Conv1d(in_channels=c_in,
                                  out_channels=c_in,
                                  kernel_size=3,
                                  padding=padding,
                                  padding_mode='circular')
        self.norm = nn.BatchNorm1d(c_in)
        self.activation = nn.ELU()
        self.maxPool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        x = self.downConv(x.permute(0, 2, 1))
        x = self.norm(x)
        x = self.activation(x)
        x = self.maxPool(x)
        x = x.transpose(1,2)
        return x

In [ ]:
class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len = 5000):
        super(PositionalEmbedding, self).__init__()
        self.max_len = max_len
        self.d_model = d_model
        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)  
        pe[:, 1::2] = torch.cos(position * div_term)  

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        
        return x + self.pe[:, :x.size(1)]

class TokenEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super(TokenEmbedding, self).__init__()
        padding = 1 if torch.__version__>='1.5.0' else 2
        self.tokenConv = nn.Conv1d(in_channels=c_in, out_channels=d_model, 
                                    kernel_size=3, padding=padding, padding_mode='circular')
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_in',nonlinearity='leaky_relu')

    def forward(self, x):
        x = self.tokenConv(x.permute(0, 2, 1)).transpose(1,2)
        return x

class DataEmbedding(nn.Module):
    def __init__(self, c_in, d_model, embed_type='fixed', freq='h', dropout=0.1):
        super(DataEmbedding, self).__init__()

        self.value_embedding = TokenEmbedding(c_in=c_in, d_model=d_model)
        #self.position_embedding = PositionalEmbedding(d_model=d_model)
        

        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = self.value_embedding(x) #+ self.position_embedding(x) 
        
        return self.dropout(x)

In [ ]:
class TriangularCausalMask():
    def __init__(self, B, L, device="cpu"):
        mask_shape = [B, 1, L, L]
        with torch.no_grad():
            self._mask = torch.triu(torch.ones(mask_shape, dtype=torch.bool), diagonal=1).to(device)

    @property
    def mask(self):
        return self._mask

class ProbMask():
    def __init__(self, B, H, L, index, scores, device="cpu"):
        _mask = torch.ones(L, scores.shape[-1], dtype=torch.bool).to(device).triu(1)
        _mask_ex = _mask[None, None, :].expand(B, H, L, scores.shape[-1])
        indicator = _mask_ex[torch.arange(B)[:, None, None],
                             torch.arange(H)[None, :, None],
                             index, :].to(device)
        self._mask = indicator.view(scores.shape).to(device)
    
    @property
    def mask(self):
        return self._mask

In [ ]:
class ProbAttention(nn.Module):
    def __init__(self, mask_flag=True, factor=5, scale=None, attention_dropout=0.1, output_attention=False):
        super(ProbAttention, self).__init__()
        self.factor = factor
        self.scale = scale
        self.mask_flag = mask_flag
        self.output_attention = output_attention
        self.dropout = nn.Dropout(attention_dropout)

    def _prob_QK(self, Q, K, sample_k, n_top):  # n_top: c*ln(L_q)
        # Q [B, H, L, D]
        B, H, L_K, E = K.shape
        _, _, L_Q, _ = Q.shape

        # calculate the sampled Q_K
        K_expand = K.unsqueeze(-3).expand(B, H, L_Q, L_K, E)
        # real U = U_part(factor*ln(L_k))*L_q
        index_sample = torch.randint(L_K, (L_Q, sample_k))
        K_sample = K_expand[:, :, torch.arange(
            L_Q).unsqueeze(1), index_sample, :]
        Q_K_sample = torch.matmul(
            Q.unsqueeze(-2), K_sample.transpose(-2, -1)).squeeze()

        # find the Top_k query with sparisty measurement
        M = Q_K_sample.max(-1)[0] - torch.div(Q_K_sample.sum(-1), L_K)
        M_top = M.topk(n_top, sorted=False)[1]

        # use the reduced Q to calculate Q_K
        Q_reduce = Q[torch.arange(B)[:, None, None],
                   torch.arange(H)[None, :, None],
                   M_top, :]  # factor*ln(L_q)
        Q_K = torch.matmul(Q_reduce, K.transpose(-2, -1))  # factor*ln(L_q)*L_k

        return Q_K, M_top

    def _get_initial_context(self, V, L_Q):
        B, H, L_V, D = V.shape
        if not self.mask_flag:
            # V_sum = V.sum(dim=-2)
            V_sum = V.mean(dim=-2)
            contex = V_sum.unsqueeze(-2).expand(B, H,
                                                L_Q, V_sum.shape[-1]).clone()
        else:  # use mask
            # requires that L_Q == L_V, i.e. for self-attention only
            assert (L_Q == L_V)
            contex = V.cumsum(dim=-2)
        return contex

    def _update_context(self, context_in, V, scores, index, L_Q, attn_mask):
        B, H, L_V, D = V.shape

        if self.mask_flag:
            attn_mask = ProbMask(B, H, L_Q, index, scores, device=V.device)
            scores.masked_fill_(attn_mask.mask, -np.inf)

        attn = torch.softmax(scores, dim=-1)  # nn.Softmax(dim=-1)(scores)

        context_in[torch.arange(B)[:, None, None],
        torch.arange(H)[None, :, None],
        index, :] = torch.matmul(attn, V).type_as(context_in)
        if self.output_attention:
            attns = (torch.ones([B, H, L_V, L_V]) /
                     L_V).type_as(attn).to(attn.device)
            attns[torch.arange(B)[:, None, None], torch.arange(H)[
                                                  None, :, None], index, :] = attn
            return context_in, attns
        else:
            return context_in, None

    def forward(self, queries, keys, values, attn_mask, tau=None, delta=None):
        B, L_Q, H, D = queries.shape
        _, L_K, _, _ = keys.shape

        queries = queries.transpose(2, 1)
        keys = keys.transpose(2, 1)
        values = values.transpose(2, 1)

        U_part = self.factor * \
                 np.ceil(np.log(L_K)).astype('int').item()  # c*ln(L_k)
        u = self.factor * \
            np.ceil(np.log(L_Q)).astype('int').item()  # c*ln(L_q)

        U_part = U_part if U_part < L_K else L_K
        u = u if u < L_Q else L_Q

        scores_top, index = self._prob_QK(
            queries, keys, sample_k=U_part, n_top=u)

        # add scale factor
        scale = self.scale or 1. / sqrt(D)
        if scale is not None:
            scores_top = scores_top * scale
        # get the context
        context = self._get_initial_context(values, L_Q)
        # update the context with selected top_k queries
        context, attn = self._update_context(
            context, values, scores_top, index, L_Q, attn_mask)
        #print(f'context dim: {context.shape}')
        return context.contiguous(), attn


In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, dim, d_ff, heads, dropout = 0.1, activation = "relu"):
        super(EncoderLayer, self).__init__()
        self.dim_per_head = dim // heads 
        self.heads = heads
        self.attention = ProbAttention()
        self.query_projection = nn.Linear(dim, dim)
        self.key_projection = nn.Linear(dim, dim)
        self.value_projection = nn.Linear(dim, dim)
        self.conv1 = nn.Conv1d(in_channels=dim, out_channels = d_ff, kernel_size = 1)
        self.conv2 = nn.Conv1d(in_channels = d_ff, out_channels = dim, kernel_size = 1)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu if activation  == "relu" else F.gelu

    def forward(self, x, attn_mask= None):
        batch_size, seq_len, dim = x.shape
        Q = self.query_projection(x)
        K = self.key_projection(x)
        V = self.value_projection(x)

        # Reshape for multi-head: [B, H, L, D_head]
        def reshape_to_heads(tensor):
            return tensor.view(batch_size, seq_len, self.heads, self.dim_per_head).transpose(1, 2)

        Q = reshape_to_heads(Q)
        K = reshape_to_heads(K)
        V = reshape_to_heads(V)
        
        new_x, _ = self.attention(Q, K, V, attn_mask = attn_mask)
        new_x = new_x.view(batch_size,seq_len,-1)
        x = x + self.dropout(new_x)
        y = self.norm1(x)
        y = self.dropout(self.activation(self.conv1(y.transpose(-1,1))))
        y = self.dropout(self.conv2(y).transpose(-1,1))

        return self.norm2(x+y)

class Encoder(nn.Module):
    def __init__(self, num_layers, dim, heads, d_ff, dropout=0.1, use_conv=False):
        super(Encoder, self).__init__()
        
        
        self.attn_layers = nn.ModuleList([
            EncoderLayer(dim, d_ff, heads, dropout) for _ in range(num_layers)
        ])

        
        self.conv_layers = nn.ModuleList([
            nn.ConvLayer(dim) for _ in range(num_layers-1)
        ]) if use_conv else None  # If `use_conv` is True, it creates the conv layers
        
        self.norm = nn.LayerNorm(dim)  

    def forward(self, x, attn_mask=None):
        attns = []

        if self.conv_layers is not None:
            for attn_layer, conv_layer in zip(self.attn_layers, self.conv_layers):
                x = attn_layer(x)  # Self attention
                x = conv_layer(x.transpose(1, 2)).transpose(1, 2)  # Convoluzione 1D
                attns.append(attn)
            x = self.attn_layers[-1](x)  # last layer without convolution
            #attns.append(attn)
        else:
            for attn_layer in self.attn_layers:
                x = attn_layer(x)
                #attns.append(attn)

        x = self.norm(x) 
        return x

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, dim, d_ff, heads, dropout=0.1, activation="relu"):
        super(DecoderLayer, self).__init__()
        self.heads = heads
        self.dim_per_head = dim // heads
        self.dim = dim

        # Attention modules
        self.self_attention = ProbAttention()
        self.cross_attention = ProbAttention()

        # Projection layers for self-attention
        self.q_proj_self = nn.Linear(dim, dim)
        self.k_proj_self = nn.Linear(dim, dim)
        self.v_proj_self = nn.Linear(dim, dim)

        # Projection layers for cross-attention
        self.q_proj_cross = nn.Linear(dim, dim)
        self.k_proj_cross = nn.Linear(dim, dim)
        self.v_proj_cross = nn.Linear(dim, dim)

        # Feedforward layers
        self.conv1 = nn.Conv1d(in_channels=dim, out_channels=d_ff, kernel_size=1)
        self.conv2 = nn.Conv1d(in_channels=d_ff, out_channels=dim, kernel_size=1)

        # Normalization and dropout
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.norm3 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

        # Activation function
        self.activation = F.relu if activation == "relu" else F.gelu

    def forward(self, x, cross, x_mask=None, cross_mask=None):
        B, L, D = x.shape  # Batch, Seq Len, Dim

        # Self Attention
        q = self.q_proj_self(x)
        k = self.k_proj_self(x)
        v = self.v_proj_self(x)

        q = q.view(B, L, self.heads, self.dim_per_head).transpose(1, 2)
        k = k.view(B, L, self.heads, self.dim_per_head).transpose(1, 2)
        v = v.view(B, L, self.heads, self.dim_per_head).transpose(1, 2)

        sa_out, _ = self.self_attention(q, k, v, attn_mask=x_mask)
        sa_out = sa_out.transpose(1, 2).contiguous().view(B, L, D)
        x = self.norm1(x + self.dropout(sa_out))

        # Cross Attention
        q = self.q_proj_cross(x)
        k = self.k_proj_cross(cross)
        v = self.v_proj_cross(cross)

        q = q.view(B, L, self.heads, self.dim_per_head).transpose(1, 2)
        k = k.view(B, k.shape[1], self.heads, self.dim_per_head).transpose(1, 2)
        v = v.view(B, v.shape[1], self.heads, self.dim_per_head).transpose(1, 2)

        ca_out, _ = self.cross_attention(q, k, v, attn_mask=cross_mask)
        ca_out = ca_out.transpose(1, 2).contiguous().view(B, L, D)
        x = self.norm2(x + self.dropout(ca_out))

        # feedforward
        y = self.dropout(self.activation(self.conv1(x.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))
        out = self.norm3(x + y)

        return out

class Decoder(nn.Module):
    def __init__(self, dim, heads, d_ff, dropout, num_layers):
        super(Decoder, self).__init__()
        self.attn_layers = nn.ModuleList([
            DecoderLayer(dim, d_ff , heads, dropout) for _ in range(num_layers)
        ])
        
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, cross, x_mask=None, cross_mask=None):
        
        for layer in self.attn_layers:
            x = layer(x, cross, x_mask=x_mask, cross_mask=cross_mask)

        
        x = self.norm(x)

        return x

In [ ]:
class Informer(nn.Module):
    def __init__(self, dim, d_ff, heads, dropout, num_encoder_layers, num_decoder_layers, pred_len, batch_size, n_sensor, output_attention=False):
        super(Informer, self).__init__()

        self.bs = batch_size
        self.ns = n_sensor
        
        # adding posional embedding
        self.enc_embedding = DataEmbedding(c_in = dim, d_model=dim)  # Encoder con Positional Encoding
        self.dec_embedding = DataEmbedding(c_in = dim, d_model=dim)  # Decoder con Positional Encoding

        
        self.encoder = Encoder(num_layers=num_encoder_layers, dim=dim, d_ff=d_ff, heads=heads, dropout=dropout)
        self.decoder = Decoder(dim=dim, d_ff=d_ff, heads=heads, dropout=dropout, num_layers=num_decoder_layers)

        self.projection = nn.Linear(dim, pred_len)
        
        
        self.pred_len = pred_len
        self.output_attention = output_attention

    def forward(self, x_enc, x_dec,enc_self_mask=None, dec_self_mask=None, dec_enc_mask=None):

        x_enc = x_enc.view(self.bs, self.ns, -1)
        # Adding positional encoding to the encoder input
        enc_out = self.enc_embedding(x_enc) 
        enc_out = self.encoder(enc_out, attn_mask=enc_self_mask)
        
        # Adding positional encoding to the decoder inputs
        dec_out = self.dec_embedding(x_dec)  
        dec_out = self.decoder(dec_out, enc_out, x_mask=dec_self_mask, cross_mask=dec_enc_mask)
        
        # Proiezione finale
        dec_out = self.projection(dec_out)
        #print(f' shape out: {dec_out.shape}')
        return dec_out

**Model BayesianGAT-Informer**

In [ ]:
class GATInformer(nn.Module):
    def __init__(self, gat_in_features, gat_hidden_features, gat_out_features, gat_num_layers, 
                 informer_dim, d_ff, heads, dropout, num_encoder_layers, num_decoder_layers, pred_len, batch_size, n_sensor, output_attention=False):
        super(GATInformer, self).__init__()

        
        self.gat = BayesianGAT(in_features=gat_in_features, 
                               hidden_features=gat_hidden_features, 
                               out_features=gat_out_features, 
                               num_layers=gat_num_layers, 
                               heads=heads, 
                               dropout=dropout)

        
        self.inf = Informer(informer_dim, d_ff, heads, dropout, num_encoder_layers, num_decoder_layers, pred_len, batch_size, n_sensor, output_attention)

    def forward(self, x_enc, x_dec, edge_index,enc_self_mask=None, dec_self_mask=None, dec_enc_mask=None):
       
        gat_out = self.gat(x_enc, edge_index)

        #
        dec_out = self.inf(gat_out, x_dec )

        return dec_out

In [ ]:
model = GATInformer(gat_in_features=int(12), gat_hidden_features = int(64), gat_out_features = int(12), gat_num_layers = int(1), informer_dim = int(12), d_ff = int(512), heads= int(4), dropout = 0.1, num_encoder_layers = 6, num_decoder_layers = 6, pred_len = 12, batch_size = batch_size, n_sensor = 207)

model.to(device)

criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
scaler = StandardScaler()
input_len = 12
target_len = 12
batch_size = 128

#generates the batches 
batched_inputs, batched_targets = generate_sliding_window_batches(train_data, input_len, target_len, batch_size)
batched_val_inputs, batched_val_targets = generate_sliding_window_batches(val_data, input_len, target_len, batch_size)
standardized_train_inputs = [scaler.fit_transform(batch.reshape(-1, batch.shape[-1])).reshape(batch.shape) for batch in batched_inputs]

In [ ]:
epochs = 100
n_sensor = 207
optimizer = optim.Adam(model.parameters(), lr=0.001)
mae_loss = nn.L1Loss()

Loss =  []
Val_loss = []
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for inputs_batch, targets_batch in zip(standardized_train_inputs, batched_targets):
        inputs_batch = torch.FloatTensor(inputs_batch).to(device)
        targets_batch = torch.FloatTensor(targets_batch).to(device)
        optimizer.zero_grad()
        x_enc = inputs_batch.view(-1, input_len)  
        batch_size = inputs_batch.shape[0]
        n_nodes = inputs_batch.shape[1]
        pred_len = targets_batch.shape[-1]

    
        pred = model(x_enc, targets_batch, edge_index)
        
        loss = mae_loss(pred, targets_batch)

        loss.backward(retain_graph=True)
        optimizer.step()
        
        total_loss += loss.item()

    avg_train_loss = total_loss / len(standardized_train_inputs)
    Loss.append(avg_train_loss)
    # Validation
    model.eval()
    val_loss = 0
    
    with torch.no_grad():
        for val_inputs_batch, val_targets_batch in zip(batched_val_inputs, batched_val_targets):
            val_inputs_batch = torch.FloatTensor(val_inputs_batch).to(device)  # [B, 207, 12]
            
            
            x_enc = val_inputs_batch.view(-1, input_len)
            batch_size = val_inputs_batch.shape[0]
            n_nodes = val_inputs_batch.shape[1]
            pred_len = val_targets_batch.shape[-1]

           
            last_step = val_inputs_batch[:,:,-1]
            x_dec = last_step.unsqueeze(-1).repeat(1,1,pred_len)
            x_enc = scaler.transform(val_inputs_batch.cpu().reshape(-1, val_inputs_batch.shape[-1])).reshape(val_inputs_batch.shape)
            x_enc = torch.FloatTensor(x_enc)  # [B, 207, 12]
            x_enc = x_enc.view(-1, x_enc.shape[-1]).to(device)
            pred = model(x_enc,x_dec,  edge_index)
            pred = pred.reshape(-1, pred.shape[-1]).cpu()
           
            val_targets_batch = val_targets_batch.reshape(-1, val_targets_batch.shape[-1])
            val_targets_batch = torch.FloatTensor(val_targets_batch)
            
            val_loss += mae_loss(pred, val_targets_batch).item()
    
    avg_val_loss = val_loss / len(batched_val_inputs)
    Val_loss.append(avg_val_loss)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")


In [ ]:
test_batched_inputs, test_batched_targets = generate_sliding_window_batches(test_data, input_len, target_len, batch_size)

In [ ]:
model.eval()
mae_loss = nn.L1Loss()
val_loss = 0

with torch.no_grad():
    for val_inputs_batch, val_targets_batch in zip(test_batched_inputs,test_batched_targets):
        val_inputs_batch = torch.FloatTensor(val_inputs_batch).to(device)
        val_targets_batch = torch.FloatTensor(val_targets_batch).to(device)
        x_enc = val_inputs_batch.to(device)
        target = val_targets_batch.to(device)
        x_enc = x_enc.view(-1, input_len)
        last_step = val_inputs_batch[:,:,-1]
        x_dec = last_step.unsqueeze(-1).repeat(1,1,12)
        x_enc = scaler.transform(val_inputs_batch.cpu().reshape(-1, val_inputs_batch.shape[-1])).reshape(val_inputs_batch.shape)
        x_enc = torch.FloatTensor(x_enc)  # [B, 207, 12]
        x_enc = x_enc.view(-1, x_enc.shape[-1]).to(device)
        pred = model(x_enc,x_dec,  edge_index)
        pred = pred.reshape(-1, pred.shape[-1]).cpu()
        
        
        val_targets_batch = val_targets_batch.reshape(-1, val_targets_batch.shape[-1]).cpu()
        
        val_loss += mae_loss(pred, val_targets_batch).item()

avg_val_loss = val_loss / len(batched_inputs)
print(avg_val_loss)